In [1]:
from os.path import basename, exists


def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")

In [2]:
try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import HTML

In [4]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dct")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dat.gz")

In [5]:
try:
    import statadict
except ImportError:
    %pip install statadict

In [6]:
dct_file = "2002FemPreg.dct"
dat_file = "2002FemPreg.dat.gz"

In [7]:
from statadict import parse_stata_dict


def read_stata(dct_file, dat_file):
    stata_dict = parse_stata_dict(dct_file)
    resp = pd.read_fwf(
        dat_file,
        names=stata_dict.names,
        colspecs=stata_dict.colspecs,
        compression="gzip",
    )
    return resp

In [9]:
preg = read_stata(dct_file, dat_file)
preg.head(5)

,caseid,pregordr,howpreg_n,howpreg_p,moscurrp,nowprgdk,pregend1,pregend2,nbrnaliv,multbrth,...,poverty_i,laborfor_i,religion_i,metro_i,basewgt,adj_mod_basewgt,finalwgt,secu_p,sest,cmintvw
0,1,1,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,3410.389399,3869.349602,6448.271112,2,9,1231
1,1,2,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,3410.389399,3869.349602,6448.271112,2,9,1231
2,2,1,NaN,NaN,NaN,NaN,5.0,NaN,3.0,5.0,...,0,0,0,0,7226.301740,8567.549110,12999.542264,2,12,1231
3,2,2,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,7226.301740,8567.549110,12999.542264,2,12,1231
4,2,3,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,7226.301740,8567.549110,12999.542264,2,12,1231


In [10]:
preg.shape

(13593, 243)

This dataset has 243 variables with information about 13,593 pregnancies

In [11]:
preg.columns

Index(['caseid', 'pregordr', 'howpreg_n', 'howpreg_p', 'moscurrp', 'nowprgdk',
       'pregend1', 'pregend2', 'nbrnaliv', 'multbrth',
       ...
       'poverty_i', 'laborfor_i', 'religion_i', 'metro_i', 'basewgt',
       'adj_mod_basewgt', 'finalwgt', 'secu_p', 'sest', 'cmintvw'],
      dtype='str', length=243)

In [12]:
pregordr = preg["pregordr"]
type(pregordr)

pandas.Series

In [13]:
pregordr.head()

0    1
1    2
2    1
3    2
4    3
Name: pregordr, dtype: int64

In [14]:
pregordr.shape

(13593,)

The last line includes the name of the `Series` and `dtype`, which is the type of the values.
In this example, `int64` indicates that the values are 64-bit integers.

The NSFG dataset contains 243 variables in total.
Here are some of the ones we'll use for the explorations in this book.

-   `caseid` is the integer ID of the respondent.

-   `pregordr` is a pregnancy serial number: the code for a respondent's first pregnancy is 1, for the second pregnancy is 2, and so on.

-   `prglngth` is the integer duration of the pregnancy in weeks.

-   `outcome` is an integer code for the outcome of the pregnancy. The code 1 indicates a live birth.

-   `birthord` is a serial number for live births: the code for a respondent's first child is 1, and so on. For outcomes other than live birth, this field is blank.

-   `birthwgt_lb` and `birthwgt_oz` contain the pounds and ounces parts of the birth weight of the baby.

-   `agepreg` is the mother's age at the end of the pregnancy.

-   `finalwgt` is the statistical weight associated with the respondent. It is a floating-point value that indicates the number of people in the U.S. population this respondent represents.

In [15]:
def show_table(d):
    df = pd.DataFrame(d)
    return HTML(df.to_html(index=False))

In [16]:
d = {
    "Value": [1, 2, 3, 4, 5, 6, "Total"],
    "Label": [
        "LIVE BIRTH",
        "INDUCED ABORTION",
        "STILLBIRTH",
        "MISCARRIAGE",
        "ECTOPIC PREGNANCY",
        "CURRENT PREGNANCY",
        "",
    ],
    "Total": [9148, 1862, 120, 1921, 190, 352, 13593],
}

show_table(d)

Value,Label,Total
1,LIVE BIRTH,9148
2,INDUCED ABORTION,1862
3,STILLBIRTH,120
4,MISCARRIAGE,1921
5,ECTOPIC PREGNANCY,190
6,CURRENT PREGNANCY,352
Total,,13593


In [17]:
preg["outcome"].value_counts().sort_index()

outcome
1    9148
2    1862
3     120
4    1921
5     190
6     352
Name: count, dtype: int64

In [18]:
counts = preg["birthwgt_lb"].value_counts(dropna=False).sort_index()
counts

birthwgt_lb
0.0        8
1.0       40
2.0       53
3.0       98
4.0      229
5.0      697
6.0     2223
7.0     3049
8.0     1889
9.0      623
10.0     132
11.0      26
12.0      10
13.0       3
14.0       3
15.0       1
51.0       1
97.0       1
98.0       1
99.0      57
NaN     4449
Name: count, dtype: int64

In [19]:
preg["agepreg"].mean()

np.float64(2468.8151197039497)

In [20]:
preg["agepreg"] /= 100.0
preg["agepreg"].mean()

np.float64(24.6881511970395)

In [21]:
preg["birthwgt_oz"].value_counts(dropna=False).sort_index()

birthwgt_oz
0.0     1037
1.0      408
2.0      603
3.0      533
4.0      525
5.0      535
6.0      709
7.0      501
8.0      756
9.0      505
10.0     475
11.0     557
12.0     555
13.0     487
14.0     475
15.0     378
97.0       1
98.0       1
99.0      46
NaN     4506
Name: count, dtype: int64

In [22]:
preg["birthwgt_oz"] = preg["birthwgt_oz"].replace([97, 98, 99], np.nan)

In [24]:
preg["totalwgt_lb"] = preg["birthwgt_lb"] + preg["birthwgt_oz"] / 16.0
preg["totalwgt_lb"].mean()

C:\Users\choyo\AppData\Local\Temp\ipykernel_13148\1897018522.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  preg["totalwgt_lb"] = preg["birthwgt_lb"] + preg["birthwgt_oz"] / 16.0


np.float64(7.270508352693882)

In [25]:
weights = preg["totalwgt_lb"]
n = weights.count()
n

np.int64(9039)

In [26]:
mean = weights.sum() / n
mean

np.float64(7.270508352693882)

In [27]:
weights.mean()

np.float64(7.270508352693882)

In [28]:
squared_deviations = (weights - mean) ** 2

In [29]:
var = squared_deviations.sum() / n
var

np.float64(2.198076890598448)

In [30]:
weights.var()

np.float64(2.198320094503139)

In [31]:
weights.var(ddof=0)

np.float64(2.198076890598448)

In [32]:
std = np.sqrt(var)
std

np.float64(1.4825912756381807)

In [33]:
weights.std(ddof=0)

np.float64(1.4825912756381807)

In [34]:
subset = preg.query("caseid == 10229")
subset.shape

(7, 244)

In [35]:
subset["outcome"].values

array([4, 4, 4, 4, 4, 4, 1])

## Exercises

### Exercise 1.1

Select the `birthord` column from `preg`, print the value counts, and compare to results published in the  codebook at <https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Dataset_Documentation/NSFG/Cycle6Codebook-Pregnancy.pdf>.

In [37]:
birthord_var = preg['birthord']
birthord_var.count()

np.int64(9148)

### Exercise 1.2

Create a new column named `totalwgt_kg` that contains birth weight in kilograms (there are approximately 2.2 pounds per kilogram).
Compute the mean and standard deviation of the new column.

In [38]:
# Remember that weights = preg["totalwgt_lb"]

preg['totalwgt_kg'] = weights / 2.2
media_wgt_kg = preg['totalwgt_kg'].mean()
std_wgt_kg = preg['totalwgt_kg'].std()

print(f"the mean in kg is: {media_wgt_kg} and its standar desviation: {std_wgt_kg}")

the mean in kg is: 3.3047765239517646 and its standar desviation: 0.6739424060206332


C:\Users\choyo\AppData\Local\Temp\ipykernel_13148\2112655586.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  preg['totalwgt_kg'] = weights / 2.2


### Exercise 1.3

What are the pregnancy lengths for the respondent with `caseid` 2298?

In [39]:
subset = preg.query("caseid == 2298")
subset.shape

(4, 245)

What was the birth weight of the first baby born to the respondent with `caseid` 5013?
Hint: You can use `and` to check more than one condition in a query.

In [42]:
subset = preg.query("caseid == 5013")
subset.head(10)

,caseid,pregordr,howpreg_n,howpreg_p,moscurrp,nowprgdk,pregend1,pregend2,nbrnaliv,multbrth,...,religion_i,metro_i,basewgt,adj_mod_basewgt,finalwgt,secu_p,sest,cmintvw,totalwgt_lb,totalwgt_kg
5516,5013,1,NaN,NaN,NaN,NaN,5.0,NaN,1.0,NaN,...,0,0,3643.044395,4548.148695,6132.268885,1,25,1231,7.3750,3.352273
5517,5013,2,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,...,0,0,3643.044395,4548.148695,6132.268885,1,25,1231,NaN,NaN
5518,5013,3,NaN,NaN,NaN,NaN,5.0,NaN,1.0,NaN,...,0,0,3643.044395,4548.148695,6132.268885,1,25,1231,8.3125,3.778409
5519,5013,4,NaN,NaN,NaN,NaN,5.0,NaN,1.0,NaN,...,0,0,3643.044395,4548.148695,6132.268885,1,25,1231,8.1250,3.693182


In [45]:
subset = preg.query("caseid == 5013 and outcome == 1 and pregordr == 1")
subset['totalwgt_kg']

5516    3.352273
Name: totalwgt_kg, dtype: float64